<a href="https://colab.research.google.com/github/pyonan/EDP-Mini-Project/blob/main/skincare_products_eda_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
╔══════════════════════════════════════════════════════════════╗
║         EDP PROJECT: SKINCARE PRODUCTS ANALYSIS             ║
║         Topic: Skincare Products Dataset (500 rows)          ║
╚══════════════════════════════════════════════════════════════╝

LIBRARIES USED:
───────────────────────────────────────────────────────────────
1. pandas         – Data loading, cleaning, manipulation, groupby, pivot_table
2. matplotlib     – Core plotting engine (basic + advanced graphs)
3. seaborn        – Statistical visualizations (boxplot, violinplot, heatmap, pairplot)
4. numpy          – Numerical operations used in bubble chart sizing
5. dtale          – ★ UNIQUE LIBRARY: Interactive browser-based data explorer
                    (Not taught in class) – Auto-generates EDA, correlations,
                    missing value analysis, and allows live filtering/sorting.

KEY FUNCTIONS USED:
───────────────────────────────────────────────────────────────
pandas:
  • pd.read_excel()        – Load Excel dataset
  • df.isnull().sum()      – Check missing values
  • df.duplicated().sum()  – Check duplicate rows
  • df.describe()          – Summary statistics
  • df.value_counts()      – Frequency counts per category
  • df.groupby().mean()    – Aggregated averages per group
  • df.pivot_table()       – Cross-tabulation of two categorical variables
  • df.corr()              – Pearson correlation matrix
  • df.str.strip()         – Remove whitespace from string columns
  • df.str.extract()       – Extract numeric SPF value using regex
  • df.map()               – Encode binary columns (Yes/No → 1/0)

matplotlib:
  • plt.subplots()         – Create multi-panel figure layouts
  • ax.bar() / ax.barh()   – Basic vertical and horizontal bar charts ★ BASIC
  • ax.hist()              – Histogram for price distribution ★ BASIC
  • ax.pie()               – Pie/donut chart ★ BASIC
  • ax.scatter()           – Scatter plot ★ BASIC
  • ax.plot()              – Line plot for trend analysis ★ BASIC
  • ax.axvline()           – Vertical reference lines (mean/median)
  • plt.colorbar()         – Color legend for scatter plot
  • ax.stackplot()         – ★ ADVANCED: Stacked area chart
  • plt.savefig()          – Save figure to PNG file

seaborn:
  • sns.boxplot()          – Distribution boxplots ★ BASIC (via seaborn)
  • sns.violinplot()       – ★ ADVANCED: Kernel density + box combined
  • sns.heatmap()          – ★ ADVANCED: Correlation/pivot heatmap
  • sns.pairplot()         – ★ ADVANCED: Multi-variable scatterplot matrix
  • sns.set_theme()        – Global plot styling

numpy:
  • np.sqrt()              – Used to scale bubble sizes in bubble chart
  • np.polyfit()           – Fit regression trend line ★ ADVANCED
  • np.poly1d()            – Evaluate regression polynomial

dtale (UNIQUE LIBRARY ★):
  • dtale.show()           – Launch interactive web-based data explorer
  • dtale.get_instance()   – Retrieve a running dtale instance
  • Built-in features: EDA, correlation matrix, missing value analysis,
    histogram builder, scatter plots, custom charts — all in-browser
  • In Google Colab: use dtale.show(df, open_browser=False) and click
    the generated ngrok/colab link to explore interactively

GRAPH TYPES IN THIS PROJECT:
───────────────────────────────────────────────────────────────
BASIC MATPLOTLIB / SEABORN:
  ✅ Bar Chart         – Products per brand, Skin type counts
  ✅ Histogram         – Price distribution
  ✅ Pie/Donut Chart   – Product types, Cruelty free %, Fragrance %
  ✅ Scatter Plot      – Price vs Popularity
  ✅ Line Plot         – Avg rating trend across brands

ADVANCED GRAPHS:
  ✅ Violin Plot       – Rating distribution per product type
  ✅ Bubble Chart      – Brand: Avg Price vs Avg Rating (bubble=product count)
  ✅ Stacked Bar Chart – Product types stacked by skin type
  ✅ Heatmap           – Brand × Product Type avg rating
  ✅ Pairplot          – Multi-variable relationships
  ✅ Regression Plot   – Price vs Popularity with trend line
"""

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs("outputs", exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────
# ★ Change this path when running on Google Colab:
#   df = pd.read_excel('/content/skincare_products.xlsx')
df = pd.read_excel('/content/skincare_products.xlsx')

print("=" * 60)
print("     EDP PROJECT: SKINCARE PRODUCTS ANALYSIS")
print("=" * 60)

# ─────────────────────────────────────────────────────────────
# 2. DATA CLEANING & PREPROCESSING
#    Functions: str.strip(), map(), str.extract(), isnull(), duplicated()
# ─────────────────────────────────────────────────────────────
print("\n📦 SECTION 1: DATA CLEANING & PREPROCESSING")
print("-" * 50)

print(f"  Shape           : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Missing Values  : {df.isnull().sum().sum()}")
print(f"  Duplicate Rows  : {df.duplicated().sum()}")

# Strip whitespace from all string columns
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

# Binary encoding using map()
df['Cruelty_Free_Enc'] = df['Cruelty Free'].map({'Yes': 1, 'No': 0})
df['Fragrance_Enc']    = df['Fragrance'].map({'Yes': 1, 'No': 0})
df['Comedogenic_Enc']  = df['Comedogenic'].map({'Yes': 1, 'No': 0})

# Extract numeric SPF using str.extract() with regex
df['SPF_Num'] = df['SPF Level'].str.extract(r'(\d+)').astype(float)

print("  ✅ Whitespace stripped from all string columns")
print("  ✅ Encoded: Cruelty Free, Fragrance, Comedogenic → 0/1")
print("  ✅ New column: SPF_Num (numeric SPF extracted via regex)")
print(f"\n  Updated Shape   : {df.shape[0]} rows × {df.shape[1]} columns")

# ─────────────────────────────────────────────────────────────
# 3. EDA – STATISTICS
#    Functions: describe(), value_counts(), groupby().mean(), corr()
# ─────────────────────────────────────────────────────────────
print("\n\n📊 SECTION 2: EXPLORATORY DATA ANALYSIS (EDA)")
print("-" * 50)

print("\n  Numerical Summary (describe):")
print(df[['Price','Rating','Popularity Score','SPF_Num']].describe().round(2).to_string())

print("\n  Product Type counts (value_counts):")
print(df['Product Type'].value_counts().to_string())

print("\n  Brand counts:")
print(df['Brand'].value_counts().to_string())

print("\n  Skin Type counts:")
print(df['Skin Type'].value_counts().to_string())

print("\n  Budget Category counts:")
print(df['Budget Category'].value_counts().to_string())

print("\n  Avg Price by Brand (groupby + mean):")
print(df.groupby('Brand')['Price'].mean().sort_values(ascending=False).round(2).to_string())

print("\n  Avg Rating by Brand:")
print(df.groupby('Brand')['Rating'].mean().sort_values(ascending=False).round(2).to_string())

print("\n  Pivot Table – Avg Rating: Brand × Product Type:")
pivot = df.pivot_table(values='Rating', index='Brand', columns='Product Type', aggfunc='mean')
print(pivot.round(2).to_string())

print("\n  Correlation Matrix:")
corr_df = df[['Price','Rating','Popularity Score','SPF_Num',
              'Cruelty_Free_Enc','Fragrance_Enc']].corr()
print(corr_df.round(2).to_string())

print(f"\n  Cruelty-Free % : {df['Cruelty_Free_Enc'].mean()*100:.1f}%")
print(f"  Fragrance %    : {df['Fragrance_Enc'].mean()*100:.1f}%")
print(f"  Comedogenic %  : {df['Comedogenic_Enc'].mean()*100:.1f}%")

# ─────────────────────────────────────────────────────────────
# 4. DTALE – UNIQUE INTERACTIVE LIBRARY
# ─────────────────────────────────────────────────────────────
print("\n\n🔬 SECTION 3: D-TALE – UNIQUE INTERACTIVE LIBRARY")
print("-" * 50)
print("""
  D-Tale is a powerful open-source Python library that launches
  an interactive, browser-based data explorer from any DataFrame.

  It was NOT taught in class — it's a professional-grade tool
  used in industry for fast exploratory data analysis.

  Features D-Tale provides automatically:
  ✅ Interactive data table with sort/filter
  ✅ Column statistics, histograms, KDE plots
  ✅ Correlation matrix with significance tests
  ✅ Missing value analysis
  ✅ Custom chart builder (bar, pie, scatter, etc.)
  ✅ Data export and code export

  HOW TO USE IN GOOGLE COLAB:
  ──────────────────────────────────────────────────────────────
  import dtale
  import dtale.app as dtale_app
  dtale_app.ACTIVE_PORT = 40000

  d = dtale.show(df, open_browser=False)
  d.open_browser()   # generates a public link in Colab
  ──────────────────────────────────────────────────────────────
  OR simply run: dtale.show(df)  in a Jupyter Notebook
  A new browser tab will open with the full interactive explorer.
""")

# Generate dtale summary statistics to output (works headlessly)
import dtale
d = dtale.show(df, open_browser=False, ignore_duplicate=True)
print(f"  D-Tale instance created successfully.")
print(f"  D-Tale URL (local): {d.main_url()}")

# Get describe data via dtale
dtale_summary = df.describe(include='all').T
dtale_summary['missing'] = df.isnull().sum()
dtale_summary['dtype'] = df.dtypes
print("\n  D-Tale-style column summary:")
print(dtale_summary[['count','mean','std','min','max','missing','dtype']].to_string())
d.kill()

# ─────────────────────────────────────────────────────────────
# 5. VISUALIZATIONS
# ─────────────────────────────────────────────────────────────
print("\n\n🎨 SECTION 4: VISUALIZATIONS")
print("-" * 50)

# Color palette
PALETTE = ['#FF6B9D','#C44569','#F8A5C2','#FF8C94','#FFAAA5',
           '#FFD3B6','#A8E6CF','#88D8B0','#88B0D8','#B088D8']
ACCENT = '#C44569'
BG = '#FFF8FB'
sns.set_theme(style="whitegrid", font_scale=1.05)

# ── BASIC FIG 1: Bar Chart + Donut Chart ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor(BG)
fig.suptitle("Fig 1 – Product & Brand Distribution (Bar + Donut Chart)", fontweight='bold')

pt_counts = df['Product Type'].value_counts()
wedges, texts, autotexts = axes[0].pie(
    pt_counts, labels=pt_counts.index, autopct='%1.1f%%',
    colors=PALETTE[:len(pt_counts)], startangle=140,
    wedgeprops=dict(width=0.6, edgecolor='white'))
[t.set_fontsize(10) for t in texts]
axes[0].set_title("★ BASIC: Donut Chart – Product Types", fontsize=11)

brand_counts = df['Brand'].value_counts()
bars = axes[1].barh(brand_counts.index, brand_counts.values,
                    color=PALETTE[:len(brand_counts)], edgecolor='white')
axes[1].set_xlabel("Number of Products")
axes[1].set_title("★ BASIC: Horizontal Bar Chart – Products per Brand", fontsize=11)
for bar, val in zip(bars, brand_counts.values):
    axes[1].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=9)
axes[1].invert_yaxis()
axes[1].set_facecolor(BG)

plt.tight_layout()
plt.savefig("outputs/fig1_basic_bar_donut.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Fig 1 [BASIC]: Bar + Donut Chart")

# ── BASIC FIG 2: Histogram + Scatter ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle("Fig 2 – Price Histogram & Price vs Popularity Scatter", fontweight='bold')

axes[0].hist(df['Price'], bins=25, color=ACCENT, edgecolor='white', alpha=0.85)
axes[0].axvline(df['Price'].mean(), color='#2d3436', linestyle='--', linewidth=2,
                label=f"Mean: ₹{df['Price'].mean():.0f}")
axes[0].axvline(df['Price'].median(), color='#6c5ce7', linestyle='--', linewidth=2,
                label=f"Median: ₹{df['Price'].median():.0f}")
axes[0].set_xlabel("Price (₹)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("★ BASIC: Histogram – Price Distribution", fontsize=11)
axes[0].legend()
axes[0].set_facecolor(BG)

sc = axes[1].scatter(df['Price'], df['Popularity Score'],
                     c=df['Rating'], cmap='RdPu', alpha=0.7, s=55,
                     edgecolors='white', linewidths=0.4)
plt.colorbar(sc, ax=axes[1], label='Rating')
axes[1].set_xlabel("Price (₹)")
axes[1].set_ylabel("Popularity Score")
axes[1].set_title("★ BASIC: Scatter Plot – Price vs Popularity", fontsize=11)
axes[1].set_facecolor(BG)

plt.tight_layout()
plt.savefig("outputs/fig2_basic_hist_scatter.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Fig 2 [BASIC]: Histogram + Scatter Plot")

# ── BASIC FIG 3: Line Chart + Pie Chart ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)
fig.suptitle("Fig 3 – Rating Trend Line & Product Attributes Pie", fontweight='bold')

brand_rating = df.groupby('Brand')['Rating'].mean().sort_values(ascending=False)
axes[0].plot(range(len(brand_rating)), brand_rating.values,
             marker='o', color=ACCENT, linewidth=2.5, markersize=8, markerfacecolor='white',
             markeredgecolor=ACCENT, markeredgewidth=2)
axes[0].set_xticks(range(len(brand_rating)))
axes[0].set_xticklabels(brand_rating.index, rotation=45, ha='right')
axes[0].set_ylabel("Average Rating")
axes[0].set_title("★ BASIC: Line Plot – Avg Rating Across Brands", fontsize=11)
axes[0].set_facecolor(BG)
for i, (brand, val) in enumerate(brand_rating.items()):
    axes[0].annotate(f"{val:.2f}", (i, val), textcoords="offset points",
                     xytext=(0, 8), ha='center', fontsize=8)

cf_counts = df['Cruelty Free'].value_counts()
axes[1].pie(cf_counts, labels=cf_counts.index, autopct='%1.1f%%',
            colors=['#A8E6CF','#FF6B9D'], startangle=90,
            wedgeprops=dict(width=0.6, edgecolor='white'))
axes[1].set_title("★ BASIC: Pie Chart – Cruelty Free Products", fontsize=11)

plt.tight_layout()
plt.savefig("outputs/fig3_basic_line_pie.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Fig 3 [BASIC]: Line Plot + Pie Chart")

# ── ADVANCED FIG 4: Violin Plot ───────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(BG)

pt_order = df.groupby('Product Type')['Rating'].median().sort_values(ascending=False).index
sns.violinplot(data=df, x='Product Type', y='Rating', order=pt_order,
               palette=PALETTE[:8], inner='box', ax=ax, cut=0)
ax.set_title("★ ADVANCED: Violin Plot – Rating Distribution by Product Type\n"
             "(Shows kernel density + median + IQR)", fontweight='bold', fontsize=12)
ax.set_xlabel("Product Type")
ax.set_ylabel("Rating")
ax.set_facecolor(BG)
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig("outputs/fig4_advanced_violin.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Fig 4 [ADVANCED]: Violin Plot")

# ── ADVANCED FIG 5: Bubble Chart ──────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(BG)

brand_stats = df.groupby('Brand').agg(
    avg_price=('Price','mean'),
    avg_rating=('Rating','mean'),
    count=('Brand','count')
).reset_index()

bubble_sizes = np.sqrt(brand_stats['count']) * 80

scatter = ax.scatter(
    brand_stats['avg_price'], brand_stats['avg_rating'],
    s=bubble_sizes, c=range(len(brand_stats)),
    cmap='RdPu', alpha=0.8, edgecolors='white', linewidths=1.5)

for _, row in brand_stats.iterrows():
    ax.annotate(row['Brand'],
                (row['avg_price'], row['avg_rating']),
                textcoords="offset points", xytext=(0, 12),
                ha='center', fontsize=9, fontweight='bold')

ax.set_xlabel("Average Price (₹)", fontsize=12)
ax.set_ylabel("Average Rating", fontsize=12)
ax.set_title("★ ADVANCED: Bubble Chart – Avg Price vs Avg Rating\n"
             "(Bubble size = number of products per brand)", fontweight='bold', fontsize=12)
ax.set_facecolor(BG)

size_legend = [50, 60, 66]
for s in size_legend:
    ax.scatter([], [], s=np.sqrt(s)*80, c='#C44569', alpha=0.5,
               label=f'{s} products', edgecolors='white')
ax.legend(title='Product Count', loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig("outputs/fig5_advanced_bubble.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Fig 5 [ADVANCED]: Bubble Chart")

# ── ADVANCED FIG 6: Stacked Bar Chart ─────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)

stacked = df.groupby(['Product Type', 'Skin Type']).size().unstack(fill_value=0)
stacked_pct = stacked.div(stacked.sum(axis=1), axis=0) * 100

skin_types = stacked_pct.columns.tolist()
products   = stacked_pct.index.tolist()
bottoms    = [0] * len(products)
colors_stack = ['#FF6B9D','#88D8B0','#88B0D8','#FFD3B6','#B088D8']

for i, skin in enumerate(skin_types):
    vals = stacked_pct[skin].values
    bars = ax.bar(products, vals, bottom=bottoms, label=skin,
                  color=colors_stack[i], edgecolor='white', width=0.6)
    for bar, val, bot in zip(bars, vals, bottoms):
        if val > 5:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bot + val/2, f'{val:.0f}%',
                    ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    bottoms = [b + v for b, v in zip(bottoms, vals)]

ax.set_xlabel("Product Type")
ax.set_ylabel("Percentage (%)")
ax.set_title("★ ADVANCED: Stacked Bar Chart – Skin Type Distribution per Product Type\n"
             "(100% stacked — shows proportional breakdown)", fontweight='bold', fontsize=12)
ax.legend(title='Skin Type', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.tick_params(axis='x', rotation=20)
ax.set_facecolor(BG)

plt.tight_layout()
plt.savefig("outputs/fig6_advanced_stacked_bar.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Fig 6 [ADVANCED]: Stacked Bar Chart")

# ── ADVANCED FIG 7: Correlation Heatmap ───────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor(BG)

corr_cols   = ['Price','Rating','Popularity Score','SPF_Num',
               'Cruelty_Free_Enc','Fragrance_Enc','Comedogenic_Enc']
corr_labels = ['Price','Rating','Popularity','SPF',
               'Cruelty-Free','Fragrance','Comedogenic']
cm = df[corr_cols].corr()
cm.index   = corr_labels
cm.columns = corr_labels

sns.heatmap(cm, annot=True, fmt='.2f', cmap='RdPu',
            linewidths=0.5, linecolor='white', ax=ax,
            vmin=-1, vmax=1, cbar_kws={'label': 'Pearson r'})
ax.set_title("★ ADVANCED: Correlation Heatmap – All Numerical Features\n"
             "(Pearson correlation; values close to ±1 = strong relationship)",
             fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig("outputs/fig7_advanced_heatmap.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Fig 7 [ADVANCED]: Correlation Heatmap")

# ── ADVANCED FIG 8: Regression / Trend Line ───────────────────
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG)

x = df['Price'].values
y = df['Popularity Score'].values

ax.scatter(x, y, c=df['Rating'], cmap='RdPu', alpha=0.6,
           s=50, edgecolors='white', linewidths=0.4, zorder=2)

# Regression line using numpy polyfit
coeffs  = np.polyfit(x, y, 1)
poly_fn = np.poly1d(coeffs)
x_line  = np.linspace(x.min(), x.max(), 200)
ax.plot(x_line, poly_fn(x_line), color='#C44569', linewidth=2.5,
        linestyle='--', label=f'Trend: y = {coeffs[0]:.4f}x + {coeffs[1]:.1f}', zorder=3)

ax.set_xlabel("Price (₹)", fontsize=12)
ax.set_ylabel("Popularity Score", fontsize=12)
ax.set_title("★ ADVANCED: Regression Plot – Price vs Popularity Score\n"
             "(np.polyfit trend line showing linear relationship)",
             fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.set_facecolor(BG)
plt.colorbar(ax.collections[0], ax=ax, label='Rating')

plt.tight_layout()
plt.savefig("outputs/fig8_advanced_regression.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Fig 8 [ADVANCED]: Regression Plot with Trend Line")

# ── ADVANCED FIG 9: Pairplot ──────────────────────────────────
print("  ⏳ Generating pairplot (takes a moment)...")
pairplot_df = df[['Price','Rating','Popularity Score','SPF_Num','Skin Type']].copy()
pairplot_df = pairplot_df.rename(columns={'Popularity Score':'Popularity','SPF_Num':'SPF'})

g = sns.pairplot(pairplot_df, hue='Skin Type', palette=PALETTE[:5],
                 plot_kws={'alpha': 0.5, 's': 20},
                 diag_kind='kde', corner=False)
g.fig.suptitle("★ ADVANCED: Pairplot – Relationships Between All Numerical Variables\n"
               "(Diagonal = KDE distribution; Off-diagonal = Scatter by Skin Type)",
               y=1.02, fontweight='bold', fontsize=11)

plt.savefig("outputs/fig9_advanced_pairplot.png", dpi=130, bbox_inches='tight')
plt.close()
print("  ✅ Fig 9 [ADVANCED]: Pairplot (Multi-variable scatterplot matrix)")

# ── ADVANCED FIG 10: Brand × Product Type Heatmap ─────────────
fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(BG)

pivot2 = df.pivot_table(values='Rating', index='Brand', columns='Product Type', aggfunc='mean')
sns.heatmap(pivot2, annot=True, fmt='.1f', cmap='RdPu',
            linewidths=0.5, linecolor='white', ax=ax, cbar_kws={'label': 'Avg Rating'})
ax.set_title("★ ADVANCED: Heatmap – Avg Rating by Brand × Product Type\n"
             "(pivot_table + heatmap – shows best brand-product combinations)",
             fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig("outputs/fig10_advanced_pivot_heatmap.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Fig 10 [ADVANCED]: Pivot Heatmap – Brand × Product Type")

# ─────────────────────────────────────────────────────────────
# 6. KEY INSIGHTS
# ─────────────────────────────────────────────────────────────
print("\n\n📝 SECTION 5: KEY INSIGHTS SUMMARY")
print("-" * 50)

top_brand        = df['Brand'].value_counts().idxmax()
most_expensive   = df.groupby('Brand')['Price'].mean().idxmax()
best_rated_brand = df.groupby('Brand')['Rating'].mean().idxmax()
most_popular_pt  = df['Product Type'].value_counts().idxmax()
best_budget      = df.groupby('Budget Category')['Rating'].mean().idxmax()

print(f"  1.  Largest brand        : {top_brand} ({df['Brand'].value_counts().max()} products)")
print(f"  2.  Most expensive brand : {most_expensive} (avg ₹{df.groupby('Brand')['Price'].mean().max():.0f})")
print(f"  3.  Highest rated brand  : {best_rated_brand} ({df.groupby('Brand')['Rating'].mean().max():.2f}/5)")
print(f"  4.  Most common product  : {most_popular_pt}")
print(f"  5.  Best rated budget    : {best_budget} tier")
print(f"  6.  Cruelty-Free         : {df['Cruelty_Free_Enc'].mean()*100:.1f}%")
print(f"  7.  Has Fragrance        : {df['Fragrance_Enc'].mean()*100:.1f}%")
print(f"  8.  Is Comedogenic       : {df['Comedogenic_Enc'].mean()*100:.1f}%")
print(f"  9.  Price range          : ₹{df['Price'].min()} – ₹{df['Price'].max()}")
print(f"  10. Rating range         : {df['Rating'].min()} – {df['Rating'].max()}")
print(f"  11. Corr(Price, Rating)  : {df['Price'].corr(df['Rating']):.3f} (very weak → price ≠ quality)")
print(f"  12. Corr(Price, Pop.)    : {df['Price'].corr(df['Popularity Score']):.3f} (weak → price ≠ popularity)")

print("\n✅ All 10 figures saved to outputs/ folder.")
print("=" * 60)

# ─────────────────────────────────────────────────────────────
# 7. PRODUCT FILTER SYSTEM
#    Takes user input to filter products matching their choices.
#    Functions: input(), df[condition], .isin(), .between()
# ─────────────────────────────────────────────────────────────
print("\n\n🔍 SECTION 6: PRODUCT FILTER SYSTEM")
print("=" * 60)
print("  Find skincare products that match YOUR preferences.")
print("  Press ENTER to skip any filter and include all options.")
print("=" * 60)

filtered = df.copy()

# ── Filter 1: Skin Type ───────────────────────────────────────
print("\n  Available Skin Types:", ', '.join(sorted(df['Skin Type'].unique())))
skin_input = input("  Enter your Skin Type (or press Enter to skip): ").strip()
if skin_input:
    filtered = filtered[filtered['Skin Type'].str.lower() == skin_input.lower()]
    print(f"  ✔ Filtered to skin type: {skin_input}  →  {len(filtered)} products remaining")

# ── Filter 2: Product Type ────────────────────────────────────
print("\n  Available Product Types:", ', '.join(sorted(df['Product Type'].unique())))
product_input = input("  Enter Product Type (or press Enter to skip): ").strip()
if product_input:
    filtered = filtered[filtered['Product Type'].str.lower() == product_input.lower()]
    print(f"  ✔ Filtered to product type: {product_input}  →  {len(filtered)} products remaining")

# ── Filter 3: Budget Category ─────────────────────────────────
print("\n  Budget Categories: Low, Medium, High")
budget_input = input("  Enter Budget Category (or press Enter to skip): ").strip()
if budget_input:
    filtered = filtered[filtered['Budget Category'].str.lower() == budget_input.lower()]
    print(f"  ✔ Filtered to budget: {budget_input}  →  {len(filtered)} products remaining")

# ── Filter 4: Cruelty Free ────────────────────────────────────
cruelty_input = input("\n  Do you want Cruelty-Free products only? (yes/no or Enter to skip): ").strip().lower()
if cruelty_input in ('yes', 'y'):
    filtered = filtered[filtered['Cruelty Free'] == 'Yes']
    print(f"  ✔ Cruelty-Free filter applied  →  {len(filtered)} products remaining")
elif cruelty_input in ('no', 'n'):
    filtered = filtered[filtered['Cruelty Free'] == 'No']
    print(f"  ✔ Non-cruelty-free filter applied  →  {len(filtered)} products remaining")

# ── Filter 5: Fragrance Free ──────────────────────────────────
frag_input = input("  Do you want Fragrance-Free products only? (yes/no or Enter to skip): ").strip().lower()
if frag_input in ('yes', 'y'):
    filtered = filtered[filtered['Fragrance'] == 'No']
    print(f"  ✔ Fragrance-Free filter applied  →  {len(filtered)} products remaining")
elif frag_input in ('no', 'n'):
    filtered = filtered[filtered['Fragrance'] == 'Yes']
    print(f"  ✔ With-Fragrance filter applied  →  {len(filtered)} products remaining")

# ── Filter 6: Minimum Rating ──────────────────────────────────
rating_input = input("\n  Minimum Rating (3.0–5.0, or press Enter to skip): ").strip()
if rating_input:
    try:
        min_rating = float(rating_input)
        filtered = filtered[filtered['Rating'] >= min_rating]
        print(f"  ✔ Minimum rating {min_rating} applied  →  {len(filtered)} products remaining")
    except ValueError:
        print("  ⚠ Invalid rating entered — skipping this filter.")

# ── Filter 7: Max Price ───────────────────────────────────────
price_input = input("  Maximum Price in ₹ (e.g. 1000, or press Enter to skip): ").strip()
if price_input:
    try:
        max_price = float(price_input)
        filtered = filtered[filtered['Price'] <= max_price]
        print(f"  ✔ Max price ₹{max_price:.0f} applied  →  {len(filtered)} products remaining")
    except ValueError:
        print("  ⚠ Invalid price entered — skipping this filter.")

# ── RESULTS ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"  🎯 FILTER RESULTS: {len(filtered)} product(s) found")
print("=" * 60)

if len(filtered) == 0:
    print("\n  ❌ No products match all your selected filters.")
    print("  💡 Try relaxing one or more filters (e.g. allow higher price or skip cruelty-free).")
else:
    display_cols = ['Brand', 'Product Type', 'Skin Type', 'Price',
                    'Rating', 'Budget Category', 'Cruelty Free', 'Fragrance']
    result = filtered[display_cols].sort_values('Rating', ascending=False).reset_index(drop=True)
    result.index += 1  # start numbering from 1

    print(f"\n  Showing top results sorted by Rating (highest first):\n")
    print(result.to_string())

    avg_price  = filtered['Price'].mean()
    avg_rating = filtered['Rating'].mean()
    top_brand_f = filtered['Brand'].value_counts().idxmax() if len(filtered) > 0 else "N/A"

    print(f"\n  📊 Summary of filtered results:")
    print(f"     • Average Price   : ₹{avg_price:.0f}")
    print(f"     • Average Rating  : {avg_rating:.2f} / 5")
    print(f"     • Top Brand       : {top_brand_f}")
    print(f"     • Cruelty-Free    : {filtered['Cruelty_Free_Enc'].mean()*100:.0f}% of results")

print("\n✅ Filter system complete.")
print("=" * 60)


     EDP PROJECT: SKINCARE PRODUCTS ANALYSIS

📦 SECTION 1: DATA CLEANING & PREPROCESSING
--------------------------------------------------
  Shape           : 500 rows × 14 columns
  Missing Values  : 0
  Duplicate Rows  : 0
  ✅ Whitespace stripped from all string columns
  ✅ Encoded: Cruelty Free, Fragrance, Comedogenic → 0/1
  ✅ New column: SPF_Num (numeric SPF extracted via regex)

  Updated Shape   : 500 rows × 18 columns


📊 SECTION 2: EXPLORATORY DATA ANALYSIS (EDA)
--------------------------------------------------

  Numerical Summary (describe):
         Price  Rating  Popularity Score  SPF_Num
count   500.00  500.00            500.00    500.0
mean   1070.95    3.95             74.77     23.8
std     510.82    0.59             14.51     18.5
min     203.00    3.00             50.00      0.0
25%     658.50    3.50             63.00      0.0
50%    1064.50    3.90             75.00     30.0
75%    1481.00    4.43             86.00     30.0
max    1997.00    5.00            100.